In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def obtener_noticias_eldeforma(url_inicial, cantidad=5):
    """
    Realiza web scraping en eldeforma.com. Esta es la versión final
    corregida, que elimina de forma robusta las etiquetas (tags) de AMBAS
    plantillas (antigua y moderna) antes de la extracción.

    Args:
        url_inicial (str): URL de una noticia para empezar a buscar.
        cantidad (int): Número de noticias a extraer.

    Returns:
        pd.DataFrame: Un DataFrame con el corpus de noticias purificado.
    """
    noticias = []
    urls_visitadas = set()
    urls_a_visitar = [url_inicial]

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    disclaimer = "Importante: Recuerda que El Deforma es un sitio de entretenimiento, humor y sátira. El contenido en nuestras notas NO debe de ser tomado como una fuente real de información aún cuando algunos elementos de la nota sean parte de la realidad. La única sección en donde el contenido de las notas es 100% real es en “Increíble pero Cierto”."

    print(f"🚀 Iniciando scraping de corrección final... Objetivo: {cantidad} noticias.")

    while len(noticias) < cantidad and urls_a_visitar:
        url_actual = urls_a_visitar.pop(0)

        if url_actual in urls_visitadas:
            continue
        urls_visitadas.add(url_actual)

        try:
            time.sleep(1)
            response = requests.get(url_actual, headers=headers, timeout=20)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')

            # --- PASO 1: LIMPIEZA GLOBAL Y DEFINITIVA ---
            # Se eliminan los elementos no deseados de TODA la página.

            # *** CORRECCIÓN CLAVE: Eliminar AMBOS tipos de contenedores de tags ***
            for tag_section in soup.find_all('div', class_='td-post-tags'):
                tag_section.decompose()
            for tag_section in soup.find_all('div', class_='post-tags'):
                tag_section.decompose()

            # Eliminar los subtítulos de contenido relacionado
            for h3 in soup.find_all('h3'):
                h3.decompose()

            # --- PASO 2: EXTRACCIÓN DEL CONTENIDO YA LIMPIO ---
            titulo = None
            contenedor_articulo = None
            contenido_limpio = None

            # Identificar la estructura (moderna o antigua)
            titulo_tag = soup.find('h1', class_='tdb-title-text')
            if titulo_tag:
                titulo = titulo_tag.get_text(strip=True)
                contenedor_articulo = soup.find('div', class_='tdb-block-inner td-fix-index')
            else:
                titulo_tag_antiguo = soup.find('h1', class_='entry-title')
                if titulo_tag_antiguo:
                    titulo = titulo_tag_antiguo.get_text(strip=True)
                    contenedor_articulo = soup.find('div', class_='entry-content')

            if contenedor_articulo:
                # Extraer el texto del contenedor que ya fue purificado
                contenido_bruto = contenedor_articulo.get_text(separator='\n\n', strip=True)
                contenido_limpio = contenido_bruto.replace(disclaimer, '').strip()

            if titulo and contenido_limpio and len(contenido_limpio) > 50:
                noticias.append({'URL': url_actual, 'Titulo': titulo, 'Contenido': contenido_limpio})
                print(f"✅ Noticia {len(noticias)}/{cantidad} extraída: {titulo}")
            else:
                print(f"⚠️ Noticia omitida en {url_actual} (No se encontró contenido válido tras la limpieza).")

            # Buscar nuevos enlaces
            for link in soup.find_all('a', href=True):
                href = link['href']
                if href.startswith('https://eldeforma.com/') and href not in urls_visitadas and href not in urls_a_visitar:
                     if "/20" in href:
                        urls_a_visitar.append(href)

        except requests.RequestException as e:
            print(f"❌ Error de red en {url_actual}: {e}")
        except Exception as e:
            print(f"❌ Error inesperado procesando {url_actual}: {e}")

    return pd.DataFrame(noticias)

# --- EJECUCIÓN DEL SCRIPT ---
# Usamos la URL que reportaste para probar la corrección
url_semilla = 'https://eldeforma.com/2020/03/31/la-casa-de-papel-4-estreno-cuarta-temporada-fecha-netflix-2020/'
df_noticias = obtener_noticias_eldeforma(url_semilla, cantidad=5)

print("\n--- ✅ Corpus de Noticias Verificado y Purificado ---")
display(df_noticias)

KeyboardInterrupt: 

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import random
import re  # Importamos la librería de expresiones regulares para la limpieza
from google.colab import drive

# --- 1. CONFIGURACIÓN INICIAL ---
# Montar Google Drive
print("📂 Conectando con Google Drive...")
drive.mount('/content/drive')

# Parámetros del scraping
TARGET_ARTICLES = 1000
SAVE_BATCH_SIZE = 50
URL_INICIAL = 'https://eldeforma.com/2020/03/31/la-casa-de-papel-4-estreno-cuarta-temporada-fecha-netflix-2020/'
NOMBRE_CARPETA = 'CorpusNoticiasDeforma'
NOMBRE_ARCHIVO = 'corpus_noticias_limpio.csv' # Nuevo nombre para no sobrescribir el anterior

# Crear ruta y carpeta
ruta_carpeta = f'/content/drive/MyDrive/{NOMBRE_CARPETA}'
os.makedirs(ruta_carpeta, exist_ok=True)
ruta_csv = os.path.join(ruta_carpeta, NOMBRE_ARCHIVO)
print(f"📁 Carpeta de trabajo: '{ruta_carpeta}'")

# --- 2. CARGAR PROGRESO ANTERIOR ---
urls_visitadas = set()
try:
    # Leemos el nuevo CSV con el delimitador correcto
    df_existente = pd.read_csv(ruta_csv, sep=';')
    if 'url' in df_existente.columns:
        urls_visitadas.update(df_existente['url'].tolist())
        print(f"✅ Se cargaron {len(urls_visitadas)} URLs de noticias ya guardadas.")
    else:
        print("⚠️ El CSV existente no tiene columna 'url'. Empezando de cero.")
except FileNotFoundError:
    print("📄 No se encontró un archivo CSV previo. Empezando desde cero.")

# --- 3. FUNCIÓN DE LIMPIEZA DE TEXTO ---
def limpiar_texto(texto):
    """
    Limpia el texto extraído para guardarlo en una sola línea del CSV.
    - Reemplaza saltos de línea y tabulaciones con un espacio.
    - Reduce espacios múltiples a uno solo.
    """
    # Reemplazar saltos de línea y tabulaciones por un espacio
    texto_limpio = texto.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    # Usar expresión regular para reemplazar 2 o más espacios por uno solo
    texto_limpio = re.sub(r'\s{2,}', ' ', texto_limpio)
    # Quitar espacios al principio y al final
    return texto_limpio.strip()

# --- 4. FUNCIÓN DE SCRAPING OPTIMIZADA ---
def obtener_noticias_eldeforma_masivo():
    """
    Función principal que maneja el scraping masivo, progresivo y seguro.
    """
    urls_a_visitar = [URL_INICIAL] if not urls_visitadas else list(urls_visitadas)
    random.shuffle(urls_a_visitar)

    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    disclaimer = "Importante: Recuerda que El Deforma es un sitio de entretenimiento, humor y sátira..."

    noticias_nuevas_en_lote = []

    while len(urls_visitadas) < TARGET_ARTICLES:
        if not urls_a_visitar:
            print("🕸️ ¡Se agotaron las URLs para visitar! Finalizando scraping.")
            break

        url_actual = urls_a_visitar.pop(0)

        if url_actual in urls_visitadas:
            continue

        pausa = random.uniform(1.5, 4.0)
        print(f"🔎 Visitando: {url_actual} (Pausa de {pausa:.2f}s)")
        time.sleep(pausa)

        try:
            response = requests.get(url_actual, headers=headers, timeout=25)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')

            for tag_section in soup.find_all('div', class_=['td-post-tags', 'post-tags']):
                tag_section.decompose()
            for h3 in soup.find_all('h3'):
                h3.decompose()

            titulo_tag = soup.find('h1', class_='tdb-title-text') or soup.find('h1', class_='entry-title')
            contenedor_articulo = soup.find('div', class_='tdb-block-inner td-fix-index') or soup.find('div', class_='entry-content')

            if titulo_tag and contenedor_articulo:
                titulo = limpiar_texto(titulo_tag.get_text())
                contenido_bruto = contenedor_articulo.get_text()
                contenido_sin_disclaimer = contenido_bruto.replace(disclaimer, '')
                # *** APLICAMOS LA LIMPIEZA FINAL AL TEXTO ***
                contenido_final = limpiar_texto(contenido_sin_disclaimer)

                if len(contenido_final) > 100:
                    noticias_nuevas_en_lote.append({'titulo': titulo, 'texto': contenido_final, 'url': url_actual})
                    urls_visitadas.add(url_actual)
                    print(f"    ✅ ¡Noticia extraída! Total: {len(urls_visitadas)}/{TARGET_ARTICLES}")

                    if len(noticias_nuevas_en_lote) >= SAVE_BATCH_SIZE:
                        print(f"\n💾 Guardando lote de {len(noticias_nuevas_en_lote)} noticias en Drive...")
                        df_lote = pd.DataFrame(noticias_nuevas_en_lote)
                        # *** GUARDADO CON PUNTO Y COMA Y SIN ÍNDICE ***
                        df_lote.to_csv(ruta_csv, mode='a', header=not os.path.exists(ruta_csv), index=False, sep=';')
                        noticias_nuevas_en_lote.clear()
                        print("    ...Lote guardado con éxito.\n")

            for link in soup.find_all('a', href=True):
                href = link['href']
                if href.startswith('https://eldeforma.com/') and "/20" in href:
                    if href not in urls_visitadas and href not in urls_a_visitar:
                        urls_a_visitar.append(href)

        except requests.exceptions.RequestException as e:
            print(f"    ❌ Error de red en {url_actual}: {e}. Reintentando más tarde.")
            urls_a_visitar.append(url_actual)
            time.sleep(60)
        except Exception as e:
            print(f"    ❌ Error inesperado procesando {url_actual}: {e}")

    if noticias_nuevas_en_lote:
        print(f"\n💾 Guardando el último lote de {len(noticias_nuevas_en_lote)} noticias...")
        df_lote = pd.DataFrame(noticias_nuevas_en_lote)
        # *** GUARDADO FINAL CON PUNTO Y COMA Y SIN ÍNDICE ***
        df_lote.to_csv(ruta_csv, mode='a', header=not os.path.exists(ruta_csv), index=False, sep=';')
        print("    ...Último lote guardado.")

# --- 5. EJECUCIÓN DEL PROCESO ---
print("\n--- INICIANDO PROCESO DE SCRAPING MASIVO Y LIMPIO ---")
obtener_noticias_eldeforma_masivo()
print(f"\n--- ✅ PROCESO FINALIZADO ---")
print(f"El corpus se encuentra en: {ruta_csv}")

📂 Conectando con Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Carpeta de trabajo: '/content/drive/MyDrive/CorpusNoticiasDeforma'
📄 No se encontró un archivo CSV previo. Empezando desde cero.

--- INICIANDO PROCESO DE SCRAPING MASIVO Y LIMPIO ---
🔎 Visitando: https://eldeforma.com/2020/03/31/la-casa-de-papel-4-estreno-cuarta-temporada-fecha-netflix-2020/ (Pausa de 3.91s)
    ✅ ¡Noticia extraída! Total: 1/1000
🔎 Visitando: https://eldeforma.com/2025/07/30/lady-racista-ayuda-multa-coperacion/ (Pausa de 2.16s)
    ✅ ¡Noticia extraída! Total: 2/1000
🔎 Visitando: https://eldeforma.com/2025/07/30/katy-perry-trudeau-zooey-deschanel/ (Pausa de 1.83s)
    ✅ ¡Noticia extraída! Total: 3/1000
🔎 Visitando: https://eldeforma.com/2025/07/29/jeans-america-eagles-ventas-sidney-sweeney/ (Pausa de 2.70s)
    ✅ ¡Noticia extraída! Total: 4/1000
🔎 Visitando: https://eldeforma.com/2025/07/29/america-rutina-en

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import random
import re
from google.colab import drive

# --- 1. CONFIGURACIÓN INICIAL ---
print("📂 Conectando con Google Drive...")
drive.mount('/content/drive', force_remount=True)

# Parámetros del scraping
NUEVO_TARGET = 9000
SAVE_BATCH_SIZE = 50
URL_INICIAL_FALLBACK = 'https://eldeforma.com/' # Usaremos la página principal como respaldo
NOMBRE_CARPETA = 'CorpusNoticiasDeforma'
NOMBRE_ARCHIVO = 'corpus_noticias_limpio.csv'

# Crear ruta y carpeta
ruta_carpeta = f'/content/drive/MyDrive/{NOMBRE_CARPETA}'
os.makedirs(ruta_carpeta, exist_ok=True)
ruta_csv = os.path.join(ruta_carpeta, NOMBRE_ARCHIVO)
print(f"📁 Carpeta de trabajo: '{ruta_carpeta}'")

# --- 2. CARGAR PROGRESO Y PREPARAR NUEVA BÚSQUEDA ---
urls_visitadas = set()
urls_para_iniciar_busqueda = []
num_noticias_actuales = 0

try:
    df_existente = pd.read_csv(ruta_csv, sep=';')
    if 'url' in df_existente.columns and not df_existente.empty:
        urls_guardadas = df_existente['url'].tolist()
        urls_visitadas.update(urls_guardadas)
        num_noticias_actuales = len(urls_visitadas)

        # *** CORRECCIÓN CLAVE ***
        # Tomamos las últimas 20 URLs guardadas como punto de partida para encontrar nuevos enlaces.
        urls_para_iniciar_busqueda = urls_guardadas[-20:]
        urls_para_iniciar_busqueda.append(URL_INICIAL_FALLBACK) # Añadimos la página principal
        random.shuffle(urls_para_iniciar_busqueda)

        print(f"✅ Se cargaron {num_noticias_actuales} URLs. La búsqueda de nuevos enlaces comenzará desde las más recientes.")
    else:
        urls_para_iniciar_busqueda = [URL_INICIAL_FALLBACK]
        print("⚠️ El CSV no tiene columna 'url' o está vacío. Empezando de cero.")
except FileNotFoundError:
    urls_para_iniciar_busqueda = [URL_INICIAL_FALLBACK]
    print("📄 No se encontró un archivo CSV previo. Empezando desde cero.")

# --- 3. FUNCIÓN DE LIMPIEZA DE TEXTO ---
def limpiar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto_limpio = texto.replace(';', '')
    texto_limpio = texto.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    texto_limpio = re.sub(r'\s{2,}', ' ', texto_limpio)
    return texto_limpio.strip()

# --- 4. FUNCIÓN DE SCRAPING OPTIMIZADA ---
def obtener_noticias_eldeforma_masivo(urls_a_visitar): # <-- Acepta la lista de inicio como argumento
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    disclaimer = "Importante: Recuerda que El Deforma es un sitio de entretenimiento, humor y sátira..."
    noticias_nuevas_en_lote = []

    while len(urls_visitadas) < NUEVO_TARGET:
        if not urls_a_visitar:
            print("🕸️ ¡Se agotaron las URLs para visitar! Finalizando scraping.")
            break

        url_actual = urls_a_visitar.pop(0)

        # Si la URL ya fue procesada, la saltamos.
        if url_actual in urls_visitadas:
            # Buscamos enlaces nuevos incluso en páginas ya visitadas, por si se actualizaron.
            # Esto es clave para no quedarnos sin URLs.
            try:
                response = requests.get(url_actual, headers=headers, timeout=10)
                soup = BeautifulSoup(response.content, 'html.parser')
                for link in soup.find_all('a', href=True):
                    href = link['href']
                    if href.startswith('https://eldeforma.com/') and "/20" in href:
                        if href not in urls_visitadas and href not in urls_a_visitar:
                            urls_a_visitar.append(href)
            except Exception:
                continue # Si falla, simplemente continuamos con la siguiente URL.
            continue

        pausa = random.uniform(1.5, 4.0)
        print(f"🔎 Visitando URL NUEVA: {url_actual} (Pausa de {pausa:.2f}s)")
        time.sleep(pausa)

        try:
            response = requests.get(url_actual, headers=headers, timeout=25)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')

            # Limpieza global...
            for element in soup.find_all(['div', 'h3'], class_=['td-post-tags', 'post-tags']):
                element.decompose()

            titulo_tag = soup.find('h1', class_='tdb-title-text') or soup.find('h1', class_='entry-title')
            contenedor_articulo = soup.find('div', class_='tdb-block-inner td-fix-index') or soup.find('div', class_='entry-content')

            if titulo_tag and contenedor_articulo:
                titulo = limpiar_texto(titulo_tag.get_text())
                contenido_bruto = contenedor_articulo.get_text()
                contenido_sin_disclaimer = contenido_bruto.replace(disclaimer, '')
                contenido_final = limpiar_texto(contenido_sin_disclaimer)

                if len(contenido_final) > 100:
                    noticias_nuevas_en_lote.append({'titulo': titulo, 'texto': contenido_final, 'url': url_actual})
                    urls_visitadas.add(url_actual)
                    print(f"    ✅ ¡Noticia extraída! Total: {len(urls_visitadas)}/{NUEVO_TARGET}")

                    if len(noticias_nuevas_en_lote) >= SAVE_BATCH_SIZE:
                        print(f"\n💾 Guardando lote de {len(noticias_nuevas_en_lote)} noticias...")
                        df_lote = pd.DataFrame(noticias_nuevas_en_lote)
                        df_lote.to_csv(ruta_csv, mode='a', header=False, index=False, sep=';')
                        noticias_nuevas_en_lote.clear()
                        print("    ...Lote guardado con éxito.\n")

            # Encontrar más enlaces para el futuro
            for link in soup.find_all('a', href=True):
                href = link['href']
                if href.startswith('https://eldeforma.com/') and "/20" in href:
                    if href not in urls_visitadas and href not in urls_a_visitar:
                        urls_a_visitar.append(href)

        except requests.exceptions.RequestException as e:
            print(f"    ❌ Error de red: {e}. Reintentando más tarde.")
            urls_a_visitar.append(url_actual)
            time.sleep(60)
        except Exception as e:
            print(f"    ❌ Error inesperado: {e}")

    # Guardar el último lote restante
    if noticias_nuevas_en_lote:
        print(f"\n💾 Guardando el último lote de {len(noticias_nuevas_en_lote)} noticias...")
        df_lote = pd.DataFrame(noticias_nuevas_en_lote)
        df_lote.to_csv(ruta_csv, mode='a', header=False, index=False, sep=';')
        print("    ...Último lote guardado.")

# --- 5. EJECUCIÓN DEL PROCESO ---
print("\n--- INICIANDO PROCESO DE SCRAPING MASIVO ---")
print(f"Noticias actuales: {num_noticias_actuales}")
print(f"Nuevo objetivo: {NUEVO_TARGET}")

if num_noticias_actuales < NUEVO_TARGET:
    print(f"Se buscarán hasta {NUEVO_TARGET - num_noticias_actuales} noticias nuevas.")
    # Le pasamos la lista de URLs iniciales a la función
    obtener_noticias_eldeforma_masivo(urls_para_iniciar_busqueda)
else:
    print("El objetivo de noticias ya ha sido alcanzado o superado. No se iniciará el scraping.")

print(f"\n--- ✅ PROCESO FINALIZADO ---")
df_final = pd.read_csv(ruta_csv, sep=';')
print(f"El corpus ahora tiene un total de {len(df_final)} noticias.")
print(f"Archivo actualizado en: {ruta_csv}")

📂 Conectando con Google Drive...
Mounted at /content/drive
📁 Carpeta de trabajo: '/content/drive/MyDrive/CorpusNoticiasDeforma'
✅ Se cargaron 1000 URLs. La búsqueda de nuevos enlaces comenzará desde las más recientes.

--- INICIANDO PROCESO DE SCRAPING MASIVO ---
Noticias actuales: 1000
Nuevo objetivo: 9000
Se buscarán hasta 8000 noticias nuevas.
🔎 Visitando URL NUEVA: https://eldeforma.com/ (Pausa de 2.81s)
🔎 Visitando URL NUEVA: https://eldeforma.com/2025/04/16/que-son-las-ptu/ (Pausa de 1.97s)
🔎 Visitando URL NUEVA: https://eldeforma.com/2025/04/14/lego-leon/ (Pausa de 2.31s)
🔎 Visitando URL NUEVA: https://eldeforma.com/2025/02/21/metodos-efectivos-para-ahorrar-pero-dandote-tus-gustitos/ (Pausa de 3.99s)
🔎 Visitando URL NUEVA: https://eldeforma.com/2025/02/17/elije-tu-fav-esa-es-la-cuestion/ (Pausa de 3.47s)
🔎 Visitando URL NUEVA: https://eldeforma.com/2025/01/30/contenido-no-pagado-2/ (Pausa de 3.97s)
🔎 Visitando URL NUEVA: https://eldeforma.com/2025/01/30/contenido-no-pagado/ (Pau

KeyboardInterrupt: 

In [ ]:
# --- PASO 0: INSTALAR LIBRERÍAS ---
!pip install trafilatura

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import random
import re
from google.colab import drive
from collections import deque
from urllib.parse import urljoin

# --- 1. CONFIGURACIÓN INICIAL ---
print("📂 Conectando con Google Drive...")
drive.mount('/content/drive', force_remount=True)

# --- PARÁMETROS DEL CRAWLER ---
NUEVO_TARGET = 9000
SAVE_BATCH_SIZE = 50
NOMBRE_CARPETA = 'CorpusNoticiasDeforma'
NOMBRE_ARCHIVO_DATOS = 'corpus_noticias_limpio.csv'
NOMBRE_ARCHIVO_COLA = 'urls_a_visitar.txt'

URLS_SEMILLA = [
    'https://eldeforma.com/2025/02/21/tamaulipas-habitanes-fuga-agua-xvanos/',
    'https://eldeforma.com/2025/06/14/onu-israel-iran-regano/',
    'https://eldeforma.com/2024/11/16/torreon-costco-pasteles-revendedores/',
    'https://eldeforma.com/2025/04/01/america-real-madrid-argentina-robar/',
    'https://eldeforma.com/2025/04/01/imagnees-ghibli-whats-tias/',
    'https://eldeforma.com/2024/11/13/kary-perry-vengalaalegria-entrevista/'
]

# Configuración de rutas
ruta_carpeta = f'/content/drive/MyDrive/{NOMBRE_CARPETA}'
os.makedirs(ruta_carpeta, exist_ok=True)
ruta_csv = os.path.join(ruta_carpeta, NOMBRE_ARCHIVO_DATOS)
ruta_cola = os.path.join(ruta_carpeta, NOMBRE_ARCHIVO_COLA)
print(f"📁 Carpeta de trabajo: '{ruta_carpeta}'")

# --- 2. CARGAR PROGRESO ANTERIOR ---
urls_articulos_guardados = set()
try:
    df_existente = pd.read_csv(ruta_csv, sep=';')
    if 'url' in df_existente.columns and not df_existente.empty:
        urls_articulos_guardados.update(df_existente['url'].tolist())
        print(f"✅ Se cargaron {len(urls_articulos_guardados)} URLs de artículos ya guardados.")
except FileNotFoundError:
    print("📄 No se encontró CSV de artículos. Empezando de cero.")

urls_a_visitar = deque()
try:
    with open(ruta_cola, 'r') as f:
        urls_a_visitar.extend([line.strip() for line in f if line.strip()])
    if urls_a_visitar:
        print(f"✅ Se cargaron {len(urls_a_visitar)} URLs en la cola de tareas desde el archivo.")
    else:
        urls_a_visitar.extend(URLS_SEMILLA)
        print("🟡 Cola de tareas vacía. Empezando con las URLs semilla.")
except FileNotFoundError:
    urls_a_visitar.extend(URLS_SEMILLA)
    print("📄 No se encontró archivo de cola de tareas. Empezando con las URLs semilla.")

# --- 3. FUNCIÓN DE LIMPIEZA ---
def limpiar_texto(texto):
    if not isinstance(texto, str): return ""
    texto_limpio = texto.replace(';', '').replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    texto_limpio = re.sub(r'\s{2,}', ' ', texto_limpio)
    return texto_limpio.strip()

# --- 4. FUNCIÓN PRINCIPAL DEL CRAWLER ---
def crawler_deforma():
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    disclaimer = "Importante: Recuerda que El Deforma es un sitio de entretenimiento, humor y sátira..."
    noticias_nuevas_en_lote = []
    urls_ya_escaneadas_sesion = set()

    while urls_a_visitar and len(urls_articulos_guardados) < NUEVO_TARGET:
        url_actual = urls_a_visitar.popleft()
        url_actual = url_actual.removesuffix('/amp/')

        if url_actual in urls_articulos_guardados or url_actual in urls_ya_escaneadas_sesion:
            continue

        pausa = random.uniform(2.0, 4.5)
        print(f"🔎 Procesando: {url_actual} (Pausa de {pausa:.2f}s) | Tareas: {len(urls_a_visitar)}")
        time.sleep(pausa)

        try:
            response = requests.get(url_actual, headers=headers, timeout=25, allow_redirects=True)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')
            urls_ya_escaneadas_sesion.add(url_actual)

            nuevos_enlaces_encontrados = 0
            for link in soup.find_all('a', href=True):
                href = link.get('href')
                if href:
                    # *** CORRECCIÓN CLAVE: IGNORAR ENLACES DE EMAIL ***
                    if 'mailto:' in href or '@' in href:
                        continue

                    enlace_completo = urljoin(url_actual, href)
                    enlace_limpio = enlace_completo.removesuffix('/amp/')

                    if enlace_limpio.startswith('https://eldeforma.com/') and any(y in enlace_limpio for y in ['/2022/', '/2023/', '/2024/', '/2025/']):
                        if enlace_limpio not in urls_articulos_guardados and enlace_limpio not in urls_a_visitar:
                            urls_a_visitar.append(enlace_limpio)
                            nuevos_enlaces_encontrados += 1
            if nuevos_enlaces_encontrados > 0:
                print(f"    🔗 Se añadieron {nuevos_enlaces_encontrados} nuevas URLs a la cola.")

            titulo_tag = soup.find('h1', class_=['tdb-title-text', 'entry-title'])
            contenedor_articulo = soup.find('div', class_=['tdb-block-inner td-fix-index', 'entry-content'])

            if titulo_tag and contenedor_articulo:
                print("    🔍 ¡Página identificada como artículo! Procediendo a extraer...")

                for elemento_a_quitar in contenedor_articulo.find_all(['div', 'h3'], class_=['td-post-tags', 'post-tags']):
                    elemento_a_quitar.decompose()

                titulo = limpiar_texto(titulo_tag.get_text())
                contenido_bruto = contenedor_articulo.get_text()
                contenido_sin_disclaimer = contenido_bruto.replace(disclaimer, '')
                contenido_final = limpiar_texto(contenido_sin_disclaimer)

                if len(contenido_final) > 500:
                    noticias_nuevas_en_lote.append({'titulo': titulo, 'texto': contenido_final, 'url': url_actual})
                    urls_articulos_guardados.add(url_actual)
                    print(f"    ✅ ¡Noticia VÁLIDA extraída! Total: {len(urls_articulos_guardados)}/{NUEVO_TARGET}")

                    if len(noticias_nuevas_en_lote) >= SAVE_BATCH_SIZE:
                        print(f"\n💾 Guardando lote y estado del crawler...")
                        pd.DataFrame(noticias_nuevas_en_lote).to_csv(ruta_csv, mode='a', header=False, index=False, sep=';')
                        noticias_nuevas_en_lote.clear()
                        with open(ruta_cola, 'w') as f: f.write('\n'.join(urls_a_visitar))
                        print("    ...Progreso guardado con éxito.\n")
                else:
                    print(f"    ⚠️ Contenido demasiado corto ({len(contenido_final)} caracteres). Se descarta.")

        except requests.exceptions.RequestException as e:
             # Si el error es un 404 (Not Found), simplemente lo ignoramos y no reintentamos
            if hasattr(e, 'response') and e.response is not None and e.response.status_code == 404:
                print(f"    ❌ Error 404 en {url_actual}. URL no encontrada, se descarta.")
            else:
                print(f"    ❌ Error de red procesando {url_actual}: {e}. Se reintentará más tarde.")
                urls_a_visitar.append(url_actual)
                time.sleep(30) # Pausa más corta en caso de error de red
        except Exception as e:
            print(f"    ❌ Error inesperado procesando {url_actual}: {e}")

    # Guardado Final
    if noticias_nuevas_en_lote:
        print(f"\n💾 Guardando el último lote de {len(noticias_nuevas_en_lote)} noticias...")
        pd.DataFrame(noticias_nuevas_en_lote).to_csv(ruta_csv, mode='a', header=False, index=False, sep=';')

    with open(ruta_cola, 'w') as f: f.write('\n'.join(urls_a_visitar))
    print("    ...Estado final del crawler guardado.")

# --- 5. EJECUCIÓN DEL PROCESO ---
num_actuales = len(urls_articulos_guardados)
print("\n--- INICIANDO CRAWLER HÍBRIDO Y PERSISTENTE (VERSIÓN 2) ---")
print(f"Noticias actuales: {num_actuales}")
print(f"Nuevo objetivo: {NUEVO_TARGET}")

if num_actuales < NUEVO_TARGET:
    crawler_deforma()
else:
    print("El objetivo de noticias ya ha sido alcanzado o superado.")

print(f"\n--- ✅ PROCESO FINALIZADO ---")
if os.path.exists(ruta_csv):
    df_final = pd.read_csv(ruta_csv, sep=';')
    print(f"El corpus ahora tiene un total de {len(df_final)} noticias.")
    print(f"Archivo de datos: {ruta_csv}")
    print(f"Archivo de tareas pendientes: {ruta_cola}")

📂 Conectando con Google Drive...
Mounted at /content/drive
📁 Carpeta de trabajo: '/content/drive/MyDrive/CorpusNoticiasDeforma'
✅ Se cargaron 2495 URLs de artículos ya guardados.
🟡 Cola de tareas vacía. Empezando con las URLs semilla.

--- INICIANDO CRAWLER HÍBRIDO Y PERSISTENTE (VERSIÓN 2) ---
Noticias actuales: 2495
Nuevo objetivo: 9000
    ...Estado final del crawler guardado.

--- ✅ PROCESO FINALIZADO ---
El corpus ahora tiene un total de 2495 noticias.
Archivo de datos: /content/drive/MyDrive/CorpusNoticiasDeforma/corpus_noticias_limpio.csv
Archivo de tareas pendientes: /content/drive/MyDrive/CorpusNoticiasDeforma/urls_a_visitar.txt


In [ ]:
 import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import random
import re
from google.colab import drive
from urllib.parse import urljoin

# --- 1. CONFIGURACIÓN INICIAL ---
print("📂 Conectando con Google Drive...")
drive.mount('/content/drive', force_remount=True)

# --- PARÁMETROS DEL CRAWLER ---
NUEVO_TARGET = 9000
SAVE_BATCH_SIZE = 50
MAX_PAGINAS_A_REVISAR = 5007 # El Deforma tiene aprox. 5007 páginas
NOMBRE_CARPETA = 'CorpusNoticiasDeforma'
NOMBRE_ARCHIVO_DATOS = 'corpus_noticias_limpio.csv'
NOMBRE_ARCHIVO_PROGRESO = 'crawler_progreso.txt' # Archivo para guardar la última página

# --- Configuración de rutas ---
ruta_carpeta = f'/content/drive/MyDrive/{NOMBRE_CARPETA}'
os.makedirs(ruta_carpeta, exist_ok=True)
ruta_csv = os.path.join(ruta_carpeta, NOMBRE_ARCHIVO_DATOS)
ruta_progreso = os.path.join(ruta_carpeta, NOMBRE_ARCHIVO_PROGRESO)
print(f"📁 Carpeta de trabajo: '{ruta_carpeta}'")

# --- 2. CARGAR PROGRESO ANTERIOR ---
urls_articulos_guardados = set()
try:
    df_existente = pd.read_csv(ruta_csv, sep=';')
    if 'url' in df_existente.columns:
        urls_articulos_guardados.update(df_existente['url'].tolist())
        print(f"✅ Se cargaron {len(urls_articulos_guardados)} URLs de artículos ya guardados.")
except FileNotFoundError:
    print("📄 No se encontró CSV de artículos. Empezando de cero.")

pagina_de_inicio = 1
try:
    with open(ruta_progreso, 'r') as f:
        contenido = f.read().strip()
        if contenido.isdigit():
            pagina_de_inicio = int(contenido)
            print(f"✅ Se reanudará el escaneo desde la página {pagina_de_inicio}.")
except FileNotFoundError:
    print("📄 No se encontró archivo de progreso. Empezando desde la página 1.")

# --- 3. FUNCIÓN DE LIMPIEZA ---
def limpiar_texto(texto):
    if not isinstance(texto, str): return ""
    texto_limpio = texto.replace(';', '').replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    texto_limpio = re.sub(r'\s{2,}', ' ', texto_limpio)
    return texto_limpio.strip()

# --- 4. FUNCIÓN PRINCIPAL DEL CRAWLER POR PAGINACIÓN ---
def crawler_paginador():
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    disclaimer = "Importante: Recuerda que El Deforma es un sitio de entretenimiento, humor y sátira..."
    noticias_nuevas_en_lote = []

    for page_num in range(pagina_de_inicio, MAX_PAGINAS_A_REVISAR + 1):
        if len(urls_articulos_guardados) >= NUEVO_TARGET:
            print("🏆 ¡Objetivo de noticias alcanzado! Finalizando.")
            break

        url_pagina_indice = f'https://eldeforma.com/page/{page_num}/'
        print(f"\n--- 🔎 Escaneando Página de Índice #{page_num}: {url_pagina_indice} ---")
        time.sleep(random.uniform(2.0, 4.0))

        try:
            response_indice = requests.get(url_pagina_indice, headers=headers, timeout=25)
            response_indice.raise_for_status()
            soup_indice = BeautifulSoup(response_indice.content, 'html.parser')

            # Extraer todos los enlaces de artículos de la página de índice
            enlaces_articulos = []
            # Buscamos el contenedor principal de posts que está después del título "Últimas noticias"
            contenedor_posts = soup_indice.find('div', class_='penci-wrapper-posts-content')
            if contenedor_posts:
                for titulo_h2 in contenedor_posts.find_all('h2', class_='grid-title'):
                    link_tag = titulo_h2.find('a', href=True)
                    if link_tag:
                        enlaces_articulos.append(link_tag['href'])

            print(f"    🔗 Se encontraron {len(enlaces_articulos)} enlaces de artículos en esta página.")

            # Procesar cada enlace de artículo encontrado
            for url_articulo in enlaces_articulos:
                url_limpia = url_articulo.removesuffix('/amp/')

                if url_limpia in urls_articulos_guardados:
                    print(f"    ℹ️ Omitiendo (ya guardado): {url_limpia}")
                    continue

                if len(urls_articulos_guardados) >= NUEVO_TARGET: break

                print(f"    → Procesando artículo: {url_limpia}")
                time.sleep(random.uniform(2.0, 4.5))

                try:
                    response_articulo = requests.get(url_limpia, headers=headers, timeout=25)
                    response_articulo.raise_for_status()
                    soup_articulo = BeautifulSoup(response_articulo.content, 'html.parser')

                    titulo_tag = soup_articulo.find('h1', class_=['tdb-title-text', 'entry-title'])
                    contenedor_articulo = soup_articulo.find('div', class_=['tdb-block-inner td-fix-index', 'entry-content'])

                    if titulo_tag and contenedor_articulo:
                        for elemento_a_quitar in contenedor_articulo.find_all(['div', 'h3'], class_=['td-post-tags', 'post-tags']):
                            elemento_a_quitar.decompose()

                        titulo = limpiar_texto(titulo_tag.get_text())
                        contenido_bruto = contenedor_articulo.get_text()
                        contenido_sin_disclaimer = contenido_bruto.replace(disclaimer, '')
                        contenido_final = limpiar_texto(contenido_sin_disclaimer)

                        if len(contenido_final) > 500:
                            noticias_nuevas_en_lote.append({'titulo': titulo, 'texto': contenido_final, 'url': url_limpia})
                            urls_articulos_guardados.add(url_limpia)
                            print(f"        ✅ ¡Noticia VÁLIDA extraída! Total: {len(urls_articulos_guardados)}/{NUEVO_TARGET}")

                            if len(noticias_nuevas_en_lote) >= SAVE_BATCH_SIZE:
                                print(f"\n💾 Guardando lote de {len(noticias_nuevas_en_lote)} noticias...")
                                pd.DataFrame(noticias_nuevas_en_lote).to_csv(ruta_csv, mode='a', header=False, index=False, sep=';')
                                noticias_nuevas_en_lote.clear()
                                with open(ruta_progreso, 'w') as f: f.write(str(page_num))
                                print("        ...Progreso guardado con éxito.\n")
                        else:
                            print(f"        ⚠️ Contenido demasiado corto. Se descarta.")
                    else:
                        print("        ⚠️ No se encontró título/contenido. Se descarta.")

                except Exception as e:
                    print(f"        ❌ Error procesando el artículo {url_limpia}: {e}")

        except Exception as e:
            print(f"    ❌ Error grave al procesar la página de índice #{page_num}. Se saltará. Error: {e}")

        # Guardar el progreso al final de cada página de índice
        with open(ruta_progreso, 'w') as f: f.write(str(page_num + 1))

    # Guardado Final
    if noticias_nuevas_en_lote:
        print(f"\n💾 Guardando el último lote de {len(noticias_nuevas_en_lote)} noticias...")
        pd.DataFrame(noticias_nuevas_en_lote).to_csv(ruta_csv, mode='a', header=False, index=False, sep=';')

    print("    ...Estado final guardado.")

# --- 5. EJECUCIÓN DEL PROCESO ---
num_actuales = len(urls_articulos_guardados)
print("\n--- INICIANDO CRAWLER POR PAGINACIÓN ---")
print(f"Noticias actuales: {num_actuales}")
print(f"Nuevo objetivo: {NUEVO_TARGET}")

if num_actuales < NUEVO_TARGET:
    crawler_paginador()
else:
    print("El objetivo de noticias ya ha sido alcanzado o superado.")

print(f"\n--- ✅ PROCESO FINALIZADO ---")
if os.path.exists(ruta_csv):
    df_final = pd.read_csv(ruta_csv, sep=';')
    print(f"El corpus ahora tiene un total de {len(df_final)} noticias.")
    print(f"Archivo de datos: {ruta_csv}")
    print(f"Archivo de progreso (última página): {ruta_progreso}")

Se truncaron las últimas líneas 5000 del resultado de transmisión.
    → Procesando artículo: https://eldeforma.com/2022/12/14/argentina-final-penal-arbitro/
        ✅ ¡Noticia VÁLIDA extraída! Total: 7074/9000
    → Procesando artículo: https://eldeforma.com/2022/12/14/posada-trabajo-aguinaldo-patrones-rifa/
        ✅ ¡Noticia VÁLIDA extraída! Total: 7075/9000
    → Procesando artículo: https://eldeforma.com/2022/12/14/electrorock-electrolit-concurso-bandas-el-gran-silencio/
        ✅ ¡Noticia VÁLIDA extraída! Total: 7076/9000
    ℹ️ Omitiendo (ya guardado): https://eldeforma.com/2022/12/14/karely-ruiz-posada-trabajo/
    ℹ️ Omitiendo (ya guardado): https://eldeforma.com/2022/12/14/goncalo-gonzalo-ramos-foto-video/
    ℹ️ Omitiendo (ya guardado): https://eldeforma.com/2022/12/14/que-nada-te-detenga-portero-invencible/
    ℹ️ Omitiendo (ya guardado): https://eldeforma.com/2022/12/13/bad-bunny-millones-azteca/

--- 🔎 Escaneando Página de Índice #807: https://eldeforma.com/page/807/ ---


In [ ]:
import pandas as pd
import os
import re # Importamos la librería de expresiones regulares
from google.colab import drive

# --- PASO 1: CONECTAR Y MONTAR GOOGLE DRIVE ---
print("📂 Conectando con Google Drive...")
drive.mount('/content/drive')

# --- PASO 2: DEFINIR RUTAS DE LOS ARCHIVOS ---
# Asegúrate de que estas rutas coincidan con la ubicación de tus archivos.
ruta_carpeta_deforma = '/content/drive/MyDrive/CorpusNoticiasDeforma'
ruta_corpus_deforma = os.path.join(ruta_carpeta_deforma, 'corpus_noticias_limpio.csv')
ruta_corpus_unificado = '/content/drive/MyDrive/CorpusNoticiasDeforma/corpus_unificado_es.csv'
ruta_salida = '/content/drive/MyDrive/CorpusNoticiasDeforma/corpus_unificado_es_deforma_completo.csv'

print("Rutas configuradas:")
print(f"  - Corpus El Deforma: {ruta_corpus_deforma}")
print(f"  - Corpus Unificado Base: {ruta_corpus_unificado}")
print(f"  - Archivo de Salida: {ruta_salida}")

# --- PASO 3: FUNCIÓN DE LIMPIEZA AVANZADA ---
def limpiar_texto_para_csv(texto):
    """
    Limpia el texto para asegurar que se guarde en una sola línea de CSV
    sin corromper el formato.
    - Reemplaza punto y coma (;) para evitar conflictos con el delimitador.
    - Reemplaza saltos de línea y tabulaciones con un espacio.
    - Reduce espacios múltiples a uno solo.
    """
    if not isinstance(texto, str):
        return "" # Devuelve un string vacío si el dato no es texto (ej. NaN)

    # *** CORRECCIÓN CLAVE: Eliminar el delimitador del texto ***
    texto_limpio = texto.replace(';', '')

    # Reemplazar saltos de línea y tabulaciones por un espacio
    texto_limpio = texto_limpio.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    # Usar expresión regular para reemplazar 2 o más espacios por uno solo
    texto_limpio = re.sub(r'\s{2,}', ' ', texto_limpio)
    # Quitar espacios al principio y al final
    return texto_limpio.strip()

# --- PASO 4: CARGAR, TRANSFORMAR Y UNIR LOS CORPUS ---
try:
    print("\n🔄 Cargando y procesando los corpus...")

    # Cargar el corpus de El Deforma
    df_deforma = pd.read_csv(ruta_corpus_deforma, sep=';')
    print(f"  - Se cargaron {len(df_deforma)} registros de 'El Deforma'.")

    # Adaptar el DataFrame de El Deforma
    df_deforma.rename(columns={'titulo': 'title', 'texto': 'text'}, inplace=True)
    df_deforma['label'] = 0
    df_deforma = df_deforma[['title', 'text', 'label']]

    # Cargar el corpus unificado base
    df_unificado = pd.read_csv(ruta_corpus_unificado, sep=';')
    print(f"  - Se cargaron {len(df_unificado)} registros del corpus unificado base.")

    # *** APLICAR LIMPIEZA A AMBOS CORPUS ANTES DE UNIR ***
    print("\n🧹 Aplicando limpieza de punto y coma y espacios a los textos...")
    df_deforma['title'] = df_deforma['title'].apply(limpiar_texto_para_csv)
    df_deforma['text'] = df_deforma['text'].apply(limpiar_texto_para_csv)

    df_unificado['title'] = df_unificado['title'].apply(limpiar_texto_para_csv)
    df_unificado['text'] = df_unificado['text'].apply(limpiar_texto_para_csv)
    print("  - Limpieza aplicada con éxito.")

    # Unir ambos DataFrames
    df_final = pd.concat([df_unificado, df_deforma], ignore_index=True)
    print(f"\n🔗 Corpus unificados. Total provisional: {len(df_final)} registros.")

    # --- PASO 5: LIMPIEZA FINAL DEL CORPUS UNIFICADO ---
    print("\n🧹 Realizando limpieza final de duplicados...")

    # Eliminar filas donde 'text' o 'label' sean nulos o vacíos
    df_final.dropna(subset=['text', 'label'], inplace=True)
    df_final = df_final[df_final['text'] != '']

    # Eliminar noticias con el mismo texto
    registros_antes = len(df_final)
    df_final.drop_duplicates(subset=['text'], inplace=True, keep='first')
    registros_despues = len(df_final)
    print(f"  - Se eliminaron {registros_antes - registros_despues} registros duplicados.")

    # Asegurarse de que la etiqueta sea un número entero
    df_final['label'] = df_final['label'].astype(int)

    # Mezclar todo el dataset de forma aleatoria
    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
    print("  - El corpus final ha sido mezclado aleatoriamente.")

    # --- PASO 6: GUARDAR EL RESULTADO ---
    df_final.to_csv(ruta_salida, index=False, sep=';', encoding='utf-8')

    print("\n--- ✅ ¡Proceso Finalizado con Éxito! ---")
    print(f"Total de registros únicos en el nuevo corpus: {len(df_final)}")
    print(f"El archivo final se ha guardado en: '{ruta_salida}'")

except FileNotFoundError as e:
    print(f"\n--- ❌ ERROR: Archivo no encontrado ---")
    print(f"No se pudo encontrar el archivo: {e.filename}")
    print("Por favor, verifica que la ruta sea correcta y que el archivo exista en tu Google Drive.")
except Exception as e:
    print(f"\n--- ❌ ERROR INESPERADO ---")
    print(f"Ocurrió un error durante el proceso: {e}")

📂 Conectando con Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rutas configuradas:
  - Corpus El Deforma: /content/drive/MyDrive/CorpusNoticiasDeforma/corpus_noticias_limpio.csv
  - Corpus Unificado Base: /content/drive/MyDrive/CorpusNoticiasDeforma/corpus_unificado_es.csv
  - Archivo de Salida: /content/drive/MyDrive/CorpusNoticiasDeforma/corpus_unificado_es_deforma_completo.csv

🔄 Cargando y procesando los corpus...
  - Se cargaron 9000 registros de 'El Deforma'.
  - Se cargaron 52689 registros del corpus unificado base.

🧹 Aplicando limpieza de punto y coma y espacios a los textos...
  - Limpieza aplicada con éxito.

🔗 Corpus unificados. Total provisional: 61689 registros.

🧹 Realizando limpieza final de duplicados...
  - Se eliminaron 15 registros duplicados.
  - El corpus final ha sido mezclado aleatoriamente.

--- ✅ ¡Proceso Finalizado con Éxito! ---
Total de registros únicos en el 

Había 12 noticias repetidas en esas 9mil, se tuvieron que descargar 12 nuevas...

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import random
import re
from google.colab import drive
from urllib.parse import urljoin, urlparse

# --- 1. CONFIGURACIÓN INICIAL ---
print("📂 Conectando con Google Drive...")
drive.mount('/content/drive', force_remount=True)

# --- PARÁMETROS ---
NUEVAS_A_EXTRAER = 12
NOMBRE_CARPETA = 'CorpusNoticiasDeforma'
NOMBRE_ARCHIVO_DATOS_EXISTENTE = 'corpus_noticias_limpio.csv'
NOMBRE_ARCHIVO_PROGRESO = 'crawler_progreso.txt'  # ahora guardará última URL semilla
DISABLE_SHORT_TEXT = False  # si True, permite textos cortos
MIN_LEN = 500

# Semilla: empieza desde aquí
URL_SEMILLA = "https://eldeforma.com/2026/04/07/influencers-playlist-trump-iran-fin/"

# --- RUTAS ---
ruta_carpeta = f'/content/drive/MyDrive/{NOMBRE_CARPETA}'
os.makedirs(ruta_carpeta, exist_ok=True)

ruta_csv_existente = os.path.join(ruta_carpeta, NOMBRE_ARCHIVO_DATOS_EXISTENTE)
ruta_progreso = os.path.join(ruta_carpeta, NOMBRE_ARCHIVO_PROGRESO)

# Archivo NUEVO para las 12
fecha_hoy = "2026-04-08"
stamp = fecha_hoy.replace("-", "")
ruta_csv_nuevo = os.path.join(ruta_carpeta, f'corpus_12_nuevas_desde_semilla_{stamp}.csv')

print(f"📁 Carpeta: {ruta_carpeta}")
print(f"📄 CSV existente: {ruta_csv_existente}")
print(f"🆕 CSV nuevo: {ruta_csv_nuevo}")
print(f"📌 Progreso: {ruta_progreso}")

# --- 2. CARGAR URLs YA GUARDADAS ---
urls_articulos_guardados = set()
try:
    df_existente = pd.read_csv(ruta_csv_existente, sep=';')
    if 'url' in df_existente.columns:
        urls_articulos_guardados.update(df_existente['url'].astype(str).tolist())
        print(f"✅ Cargadas {len(urls_articulos_guardados)} URLs ya guardadas.")
    else:
        print("⚠️ El CSV existente no tiene columna 'url'. Deduplicación limitada.")
except FileNotFoundError:
    print("📄 No existe CSV previo. Se continuará sin deduplicación.")
except Exception as e:
    print(f"⚠️ Error leyendo CSV existente: {e}")

# --- 3. SI QUIERES REANUDAR DESDE ÚLTIMA URL, DESCOMENTA ESTO ---
# try:
#     with open(ruta_progreso, 'r') as f:
#         last = f.read().strip()
#         if last.startswith("http"):
#             URL_SEMILLA = last
#             print(f"✅ Reanudando desde la última URL guardada en progreso: {URL_SEMILLA}")
# except FileNotFoundError:
#     pass

# --- 4. LIMPIEZA ---
def limpiar_texto(texto):
    if not isinstance(texto, str): return ""
    texto_limpio = texto.replace(';', '').replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    texto_limpio = re.sub(r'\s{2,}', ' ', texto_limpio)
    return texto_limpio.strip()

def normalize_url(u: str) -> str:
    # Quita /amp/ y fragmentos, normaliza dominio/ruta
    u = u.strip()
    u = u.replace("/amp/", "/")
    u = u.split("#")[0]
    # quitar query rara si apareciera
    u = u.split("?")[0]
    # asegurar https
    if u.startswith("http://"):
        u = "https://" + u[len("http://"):]
    return u

def is_eldeforma_article(u: str) -> bool:
    try:
        pu = urlparse(u)
        if pu.netloc not in ("eldeforma.com", "www.eldeforma.com"):
            return False
        # patrón típico: /YYYY/MM/DD/slug/
        return bool(re.match(r"^/\d{4}/\d{2}/\d{2}/[^/]+/?$", pu.path))
    except:
        return False

# --- 5. EXTRACCIÓN DE UN ARTÍCULO ---
def extraer_articulo(url, headers):
    disclaimer = "Importante: Recuerda que El Deforma es un sitio de entretenimiento, humor y sátira..."
    resp = requests.get(url, headers=headers, timeout=25)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.content, 'html.parser')

    titulo_tag = soup.find('h1', class_=['tdb-title-text', 'entry-title'])
    contenedor = soup.find('div', class_=['tdb-block-inner td-fix-index', 'entry-content'])

    if not titulo_tag or not contenedor:
        return None

    for elemento_a_quitar in contenedor.find_all(['div', 'h3'], class_=['td-post-tags', 'post-tags']):
        elemento_a_quitar.decompose()

    titulo = limpiar_texto(titulo_tag.get_text())
    contenido_bruto = contenedor.get_text()
    contenido_sin_disclaimer = contenido_bruto.replace(disclaimer, '')
    texto = limpiar_texto(contenido_sin_disclaimer)

    if (not DISABLE_SHORT_TEXT) and len(texto) <= MIN_LEN:
        return None

    return {"titulo": titulo, "texto": texto, "url": url}

# --- 6. DESCUBRIR ENLACES DE OTROS ARTÍCULOS DESDE UN POST ---
def descubrir_enlaces(soup, base_url):
    links = set()

    # a) cualquier <a href> dentro del html
    for a in soup.find_all("a", href=True):
        href = urljoin(base_url, a["href"])
        href = normalize_url(href)
        if is_eldeforma_article(href):
            links.add(href)

    return list(links)

# --- 7. CRAWLER TIPO GRAFO DESDE SEMILLA ---
def crawler_desde_semilla(url_inicio):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    cola = [normalize_url(url_inicio)]
    vistos = set()
    nuevas = []

    while cola and len(nuevas) < NUEVAS_A_EXTRAER:
        url = cola.pop(0)
        url = normalize_url(url)

        if url in vistos:
            continue
        vistos.add(url)

        # Saltar si ya existe en tu corpus
        if url in urls_articulos_guardados:
            print(f"ℹ️ Ya guardada, salto: {url}")
            continue

        print(f"\n→ Visitando: {url}")
        time.sleep(random.uniform(1.8, 3.6))

        try:
            resp = requests.get(url, headers=headers, timeout=25)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.content, "html.parser")

            art = extraer_articulo(url, headers)
            if art:
                nuevas.append(art)
                urls_articulos_guardados.add(url)
                print(f"✅ NUEVA #{len(nuevas)}/{NUEVAS_A_EXTRAER}: {art['titulo'][:90]}...")

                # Guardar progreso (última URL útil)
                try:
                    with open(ruta_progreso, "w") as f:
                        f.write(url)
                except Exception as e:
                    print(f"⚠️ No se pudo guardar progreso: {e}")
            else:
                print("⚠️ No válido (sin contenido o muy corto).")

            # Descubrir más links desde este post y agregarlos a la cola
            nuevos_links = descubrir_enlaces(soup, url)
            # Prioriza los que parecen más recientes (por path) de manera simple
            nuevos_links = sorted(set(nuevos_links), reverse=True)

            # Meter a cola los que no hemos visto todavía
            for lk in nuevos_links:
                if lk not in vistos and lk not in cola and lk not in urls_articulos_guardados:
                    cola.append(lk)

            print(f"   🔗 Cola ahora: {len(cola)} enlaces candidatos")

        except Exception as e:
            print(f"❌ Error visitando {url}: {e}")

    return nuevas

# --- 8. EJECUCIÓN + GUARDADO ---
print("\n--- INICIANDO CRAWLER DESDE SEMILLA ---")
print(f"Semilla: {URL_SEMILLA}")

nuevas = crawler_desde_semilla(URL_SEMILLA)

if nuevas:
    pd.DataFrame(nuevas).to_csv(ruta_csv_nuevo, index=False, sep=';')
    print(f"\n💾 Guardadas {len(nuevas)} noticias NUEVAS en:")
    print(ruta_csv_nuevo)
else:
    print("\n⚠️ No se logró extraer ninguna noticia nueva desde esa semilla.")

📂 Conectando con Google Drive...
Mounted at /content/drive
📁 Carpeta: /content/drive/MyDrive/CorpusNoticiasDeforma
📄 CSV existente: /content/drive/MyDrive/CorpusNoticiasDeforma/corpus_noticias_limpio.csv
🆕 CSV nuevo: /content/drive/MyDrive/CorpusNoticiasDeforma/corpus_12_nuevas_desde_semilla_20260408.csv
📌 Progreso: /content/drive/MyDrive/CorpusNoticiasDeforma/crawler_progreso.txt
✅ Cargadas 9000 URLs ya guardadas.

--- INICIANDO CRAWLER DESDE SEMILLA ---
Semilla: https://eldeforma.com/2026/04/07/influencers-playlist-trump-iran-fin/

→ Visitando: https://eldeforma.com/2026/04/07/influencers-playlist-trump-iran-fin/
✅ NUEVA #1/12: Influencers preparan playlist para el fin del mundo tras anuncio de Trump sobre Irán...
   🔗 Cola ahora: 29 enlaces candidatos

→ Visitando: https://eldeforma.com/2026/04/07/margaret-qualley-video-labios-sabrina/
✅ NUEVA #2/12: Margaret Qualley se someterá a terapia para dejar de morderse los labios...
   🔗 Cola ahora: 29 enlaces candidatos

→ Visitando: https

In [ ]:
import pandas as pd
import os
from google.colab import drive

# --- PASO 1: CONECTAR Y MONTAR GOOGLE DRIVE ---
print("📂 Conectando con Google Drive...")
# Usamos force_remount=True por si la sesión ya estaba montada
drive.mount('/content/drive', force_remount=True)

# --- PASO 2: DEFINIR RUTAS EN DRIVE ---
ruta_base = '/content/drive/MyDrive/CorpusNoticiasDeforma'
ruta_corpus_principal = os.path.join(ruta_base, 'corpus_unificado_es2026_duplicados.csv')
ruta_corpus_deforma = os.path.join(ruta_base, 'corpus_noticias_limpio.csv')
ruta_salida_3_clases = os.path.join(ruta_base, 'corpus_unificado_3_clases_final.csv')

# Establecemos la meta inamovible dictada por el artículo
TOTAL_OBJETIVO = 61674

print(f"\nIniciando la construcción del corpus de 3 clases (Objetivo estricto: {TOTAL_OBJETIVO} registros)...")

# --- PASO 3: CARGAR Y VALIDAR EL CORPUS PRINCIPAL ---
df_principal = pd.read_csv(ruta_corpus_principal, sep=';', encoding='utf-8')
df_principal = df_principal[['text', 'label']]

# Limpieza preventiva para asegurar que el conteo base es 100% real
df_principal.dropna(subset=['text', 'label'], inplace=True)
df_principal = df_principal[df_principal['text'].str.strip() != '']
df_principal['label'] = df_principal['label'].astype(int)

registros_principal = len(df_principal)
faltantes = TOTAL_OBJETIVO - registros_principal

print(f"  - Registros limpios en corpus principal (Real/Fake): {registros_principal}")
print(f"  - Registros requeridos de Sátira para alcanzar la meta exacta: {faltantes}")

# --- PASO 4: PREPARAR EL CORPUS DE SÁTIRA Y EXTRAER LA CANTIDAD EXACTA ---
df_deforma = pd.read_csv(ruta_corpus_deforma, sep=';', encoding='utf-8')

# Renombramos 'texto' a 'text' y asignamos la etiqueta 2
df_deforma = df_deforma.rename(columns={'texto': 'text'})
df_deforma['label'] = 2
df_deforma = df_deforma[['text', 'label']]

# Limpiamos nulos y duplicados INTERNOS en el corpus de sátira para asegurar calidad
df_deforma.dropna(subset=['text', 'label'], inplace=True)
df_deforma = df_deforma[df_deforma['text'].str.strip() != '']
df_deforma.drop_duplicates(subset=['text'], inplace=True)

print(f"  - Registros disponibles y únicos en El Deforma: {len(df_deforma)}")

# Extraemos EXACTAMENTE la cantidad de registros que nos faltan para llegar a los 61,674
if len(df_deforma) >= faltantes:
    df_deforma_seleccionado = df_deforma.sample(n=faltantes, random_state=42)
    print(f"  - Se seleccionaron matemáticamente {len(df_deforma_seleccionado)} registros de Sátira.")
else:
    print(f"⚠️ ADVERTENCIA CRÍTICA: No hay suficientes registros en El Deforma para alcanzar la meta.")
    df_deforma_seleccionado = df_deforma.copy()

# --- PASO 5: CONCATENAR PARA EL DATASET FINAL ---
# Como ya aseguramos los números, simplemente unimos los dataframes
df_final = pd.concat([df_principal, df_deforma_seleccionado], ignore_index=True)

# --- PASO 6: MEZCLAR EL DATASET (SHUFFLE) ---
# Mezclamos para que el modelo no lea primero todas las reales/falsas y luego toda la sátira
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
print("  - El corpus final ha sido mezclado aleatoriamente.")

# --- PASO 7: ESTADÍSTICAS Y GUARDADO ---
total_final = len(df_final)
print("\n📊 --- DISTRIBUCIÓN DE LABELS EN RESULTADO FINAL ---")
print(f"🎯 Total de registros alcanzados: {total_final} (Objetivo: {TOTAL_OBJETIVO})")

if total_final == TOTAL_OBJETIVO:
    print("✅ ¡LAS MATEMÁTICAS CUADRAN PERFECTAMENTE CON EL MANUSCRITO!")
else:
    print("❌ ERROR: El total no coincide con la meta de 61674.")

conteo = df_final['label'].value_counts().sort_index()

print(f"📰 Label 0 - FALSOS (fake news): {conteo.get(0, 0)} ({(conteo.get(0, 0)/total_final)*100:.1f}%)")
print(f"📰 Label 1 - VERDADEROS (noticias reales): {conteo.get(1, 0)} ({(conteo.get(1, 0)/total_final)*100:.1f}%)")
print(f"📰 Label 2 - SÁTIRA (El Deforma): {conteo.get(2, 0)} ({(conteo.get(2, 0)/total_final)*100:.1f}%)")

df_final.to_csv(ruta_salida_3_clases, index=False, sep=';', encoding='utf-8')

print(f"\n✅ Proceso Finalizado. Archivo listo para entrenar guardado en: '{ruta_salida_3_clases}'")

📂 Conectando con Google Drive...
Mounted at /content/drive

Iniciando la construcción del corpus de 3 clases (Objetivo estricto: 61674 registros)...
  - Registros limpios en corpus principal (Real/Fake): 52689
  - Registros requeridos de Sátira para alcanzar la meta exacta: 8985
  - Registros disponibles y únicos en El Deforma: 9000
  - Se seleccionaron matemáticamente 8985 registros de Sátira.
  - El corpus final ha sido mezclado aleatoriamente.

📊 --- DISTRIBUCIÓN DE LABELS EN RESULTADO FINAL ---
🎯 Total de registros alcanzados: 61674 (Objetivo: 61674)
✅ ¡LAS MATEMÁTICAS CUADRAN PERFECTAMENTE CON EL MANUSCRITO!
📰 Label 0 - FALSOS (fake news): 21746 (35.3%)
📰 Label 1 - VERDADEROS (noticias reales): 30943 (50.2%)
📰 Label 2 - SÁTIRA (El Deforma): 8985 (14.6%)

✅ Proceso Finalizado. Archivo listo para entrenar guardado en: '/content/drive/MyDrive/CorpusNoticiasDeforma/corpus_unificado_3_clases_final.csv'


In [5]:
!pip install cloudscraper
!pip install curl_cffi

In [10]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import time
import csv
import random
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from curl_cffi import requests as c_requests

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
BASE = "https://www.elmundotoday.com/"
TARGET_ARTICLES = 1000
MIN_TEXT_CHARS = 200
SLEEP_RANGE = (1.5, 3.0)
TIMEOUT = 15

USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/CorpusNoticiasDeforma/elmundotoday_eval"
LOCAL_DIR = "./elmundotoday_eval"

SEED_URLS = [
    "https://www.elmundotoday.com/",
    "https://www.elmundotoday.com/nacional/",
    "https://www.elmundotoday.com/internacional/",
    "https://www.elmundotoday.com/sociedad/",
    "https://www.elmundotoday.com/cultura/",
    "https://www.elmundotoday.com/deportes/",
    "https://www.elmundotoday.com/tecnologia/",
    "https://www.elmundotoday.com/gente/"
]
MAX_PAGES_PER_SECTION = 150

# ----------------------------------------------------------------------
def setup_output():
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as e:
            print(f"[WARN] No se pudo montar Drive ({e}); uso ruta local.")
            return LOCAL_DIR
        os.makedirs(DRIVE_DIR, exist_ok=True)
        return DRIVE_DIR
    os.makedirs(LOCAL_DIR, exist_ok=True)
    return LOCAL_DIR

def get_soup(url):
    try:
        r = c_requests.get(url, impersonate="chrome", timeout=TIMEOUT)
        if r.status_code != 200:
            print(f"  [skip] {url} (Error {r.status_code})")
            return None
        return BeautifulSoup(r.content, "html.parser")
    except Exception as e:
        print(f"  [skip] {url} ({e})")
        return None

def is_article_url(href):
    if not href:
        return False
    p = urlparse(href)
    if "elmundotoday.com" not in p.netloc:
        return False
    path = p.path
    if re.match(r"^/\d{4}/\d{2}/.+", path):
        if any(x in path for x in ("/page/", "/feed", "/tag/", "/category/", "/author/", "#")):
            return False
        return True
    return False

def collect_article_links(existing_urls, target_needed):
    """Recolecta enlaces filtrando los que ya tenemos en el CSV."""
    found = set()

    for seed in SEED_URLS:
        for page in range(1, MAX_PAGES_PER_SECTION + 1):
            if len(found) >= target_needed + 100: # Margen extra
                break

            url = seed if page == 1 else urljoin(seed, f"page/{page}/")
            soup = get_soup(url)
            if soup is None:
                break

            new_here = 0
            for a in soup.find_all("a", href=True):
                href = a.get("href")
                if href:
                    full = urljoin(BASE, href)
                    # Comprobamos que sea artículo y que NO esté en el checkpoint ni en los encontrados ahora
                    if is_article_url(full) and full not in existing_urls and full not in found:
                        found.add(full)
                        new_here += 1

            print(f"[seeds] {url} -> +{new_here} nuevos (total candidatos esta sesión: {len(found)})")

            if new_here == 0 and page > 1:
                break

            time.sleep(random.uniform(*SLEEP_RANGE))

        if len(found) >= target_needed + 100:
            break

    return list(found)

def extract_article(url):
    soup = get_soup(url)
    if soup is None:
        return None

    h1 = soup.find("h1")
    titulo = h1.get_text(strip=True) if h1 else (
        soup.find("title").get_text(strip=True) if soup.find("title") else "")

    parrafos = soup.find_all("p")
    texto = " ".join(" ".join(p.stripped_strings) for p in parrafos)

    # Limpieza
    texto = re.sub(r"Iniciar sesión para dejar un comentario.*", "", texto, flags=re.IGNORECASE)
    texto = re.sub(r"©\s*El Mundo Today.*", "", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\s+", " ", texto).strip()

    if len(texto) < MIN_TEXT_CHARS:
        return None
    return titulo, texto

def main():
    out_dir = setup_output()
    out_csv = os.path.join(out_dir, "elmundotoday_satire_eval.csv")
    print(f"[INFO] Archivo objetivo: {out_csv}\n")

    # --- FASE 0: CHECKPOINT ---
    existing_urls = set()
    seen_titles = set()
    scraped_count = 0
    file_exists = os.path.exists(out_csv)

    if file_exists:
        try:
            with open(out_csv, "r", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    existing_urls.add(row["url"])
                    seen_titles.add(row["titulo"].lower().strip())
                    scraped_count += 1
            print(f"[CHECKPOINT] Se encontraron {scraped_count} artículos ya guardados en el CSV.")
        except Exception as e:
            print(f"[ERROR] No se pudo leer el CSV existente ({e}). Empezando de cero.")
            file_exists = False
            existing_urls = set()
            seen_titles = set()
            scraped_count = 0
    else:
        print("[CHECKPOINT] No hay CSV previo. Empezando de cero.")

    articles_needed = TARGET_ARTICLES - scraped_count

    if articles_needed <= 0:
        print("\n[OK] ¡Ya has alcanzado o superado el objetivo de 1000 artículos!")
        return

    print(f"[INFO] Faltan {articles_needed} artículos por recolectar.\n")

    # --- FASE 1: ENLACES ---
    print("[1/2] Recolectando enlaces nuevos...")
    links = collect_article_links(existing_urls, articles_needed)
    random.shuffle(links)
    print(f"[INFO] {len(links)} enlaces nuevos candidatos.\n")

    if not links:
        print("[INFO] No se encontraron nuevos enlaces. Terminando.")
        return

    # --- FASE 2: EXTRACCIÓN Y GUARDADO INCREMENTAL ---
    print("[2/2] Extrayendo artículos y guardando en tiempo real...")

    # Abrimos en modo "a" (append) para añadir al final del archivo
    with open(out_csv, "a", newline="", encoding="utf-8") as f:
        fieldnames = ["titulo", "texto", "label", "fuente", "url"]
        w = csv.DictWriter(f, fieldnames=fieldnames)

        # Escribimos cabecera solo si el archivo es nuevo
        if not file_exists:
            w.writeheader()
            file_exists = True # Por si acaso

        session_scraped = 0
        for i, url in enumerate(links, 1):
            if (scraped_count + session_scraped) >= TARGET_ARTICLES:
                break

            res = extract_article(url)
            if res is None:
                continue

            titulo, texto = res
            key = titulo.lower().strip()
            if key in seen_titles:
                continue

            seen_titles.add(key)
            existing_urls.add(url)

            # Escribir la fila directamente en el disco
            w.writerow({
                "titulo": titulo,
                "texto": texto,
                "label": 2,
                "fuente": "ElMundoToday",
                "url": url,
            })

            # ¡CLAVE PARA COLAB! Forzamos la escritura física en Drive
            f.flush()
            os.fsync(f.fileno())

            session_scraped += 1
            total_current = scraped_count + session_scraped

            if session_scraped % 10 == 0 or total_current == TARGET_ARTICLES:
                print(f"  Progreso total: {total_current}/{TARGET_ARTICLES} (Guardados en esta sesión: {session_scraped})")

            time.sleep(random.uniform(*SLEEP_RANGE))

    print(f"\n[OK] Extracción finalizada. Tienes {scraped_count + session_scraped} artículos seguros en Drive.")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[INFO] Archivo objetivo: /content/drive/MyDrive/CorpusNoticiasDeforma/elmundotoday_eval/elmundotoday_satire_eval.csv

[CHECKPOINT] No hay CSV previo. Empezando de cero.
[INFO] Faltan 1000 artículos por recolectar.

[1/2] Recolectando enlaces nuevos...
[seeds] https://www.elmundotoday.com/ -> +34 nuevos (total candidatos esta sesión: 34)
[seeds] https://www.elmundotoday.com/page/2/ -> +3 nuevos (total candidatos esta sesión: 37)
[seeds] https://www.elmundotoday.com/page/3/ -> +3 nuevos (total candidatos esta sesión: 40)
[seeds] https://www.elmundotoday.com/page/4/ -> +3 nuevos (total candidatos esta sesión: 43)
[seeds] https://www.elmundotoday.com/page/5/ -> +3 nuevos (total candidatos esta sesión: 46)
[seeds] https://www.elmundotoday.com/page/6/ -> +3 nuevos (total candidatos esta sesión: 49)
[seeds] https://www.elmundotoday.com/page/7/ -> +3 nuevos (total ca

In [11]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import time
import csv
import random
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from curl_cffi import requests as c_requests

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
BASE = "https://www.elmundotoday.com/"
TARGET_ARTICLES = 1000
MIN_TEXT_CHARS = 200
SLEEP_RANGE = (1.5, 3.0)
TIMEOUT = 15

USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/CorpusNoticiasDeforma/elmundotoday_eval"
LOCAL_DIR = "./elmundotoday_eval"

# NUEVAS SEMILLAS EXPANDIDAS PARA LLEGAR A LOS 1000
SEED_URLS = [
    # Categorías principales
    "https://www.elmundotoday.com/",
    "https://www.elmundotoday.com/nacional/",
    "https://www.elmundotoday.com/internacional/",
    "https://www.elmundotoday.com/sociedad/",
    "https://www.elmundotoday.com/cultura/",
    "https://www.elmundotoday.com/deportes/",
    "https://www.elmundotoday.com/tecnologia/",
    "https://www.elmundotoday.com/gente/",
    "https://www.elmundotoday.com/ciencia/",
    "https://www.elmundotoday.com/television/",
    "https://www.elmundotoday.com/economia/",
    "https://www.elmundotoday.com/comunicados/",

    # Etiquetas (Tags) muy populares que agrupan cientos de artículos antiguos
    "https://www.elmundotoday.com/tag/politica/",
    "https://www.elmundotoday.com/tag/espana/",
    "https://www.elmundotoday.com/tag/madrid/",
    "https://www.elmundotoday.com/tag/rey/",
    "https://www.elmundotoday.com/tag/pedro-sanchez/",
    "https://www.elmundotoday.com/tag/internet/",
    "https://www.elmundotoday.com/tag/trabajo/",
    "https://www.elmundotoday.com/tag/amor/",
    "https://www.elmundotoday.com/tag/sexo/",
    "https://www.elmundotoday.com/tag/familia/",
    "https://www.elmundotoday.com/tag/salud/",
    "https://www.elmundotoday.com/tag/animales/",
    "https://www.elmundotoday.com/tag/iglesia/",
    "https://www.elmundotoday.com/tag/cataluna/",
    "https://www.elmundotoday.com/tag/andalucia/",
    "https://www.elmundotoday.com/tag/policia/"
]
MAX_PAGES_PER_SECTION = 150

# ----------------------------------------------------------------------
def setup_output():
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as e:
            print(f"[WARN] No se pudo montar Drive ({e}); uso ruta local.")
            return LOCAL_DIR
        os.makedirs(DRIVE_DIR, exist_ok=True)
        return DRIVE_DIR
    os.makedirs(LOCAL_DIR, exist_ok=True)
    return LOCAL_DIR

def get_soup(url):
    try:
        r = c_requests.get(url, impersonate="chrome", timeout=TIMEOUT)
        if r.status_code != 200:
            print(f"  [skip] {url} (Error {r.status_code})")
            return None
        return BeautifulSoup(r.content, "html.parser")
    except Exception as e:
        print(f"  [skip] {url} ({e})")
        return None

def is_article_url(href):
    if not href:
        return False
    p = urlparse(href)
    if "elmundotoday.com" not in p.netloc:
        return False
    path = p.path
    if re.match(r"^/\d{4}/\d{2}/.+", path):
        if any(x in path for x in ("/page/", "/feed", "/tag/", "/category/", "/author/", "#")):
            return False
        return True
    return False

def collect_article_links(existing_urls, target_needed):
    """Recolecta enlaces filtrando los que ya tenemos en el CSV."""
    found = set()

    for seed in SEED_URLS:
        for page in range(1, MAX_PAGES_PER_SECTION + 1):
            if len(found) >= target_needed + 100: # Margen extra
                break

            url = seed if page == 1 else urljoin(seed, f"page/{page}/")
            soup = get_soup(url)
            if soup is None:
                break

            new_here = 0
            for a in soup.find_all("a", href=True):
                href = a.get("href")
                if href:
                    full = urljoin(BASE, href)
                    # Comprobamos que sea artículo y que NO esté en el checkpoint ni en los encontrados ahora
                    if is_article_url(full) and full not in existing_urls and full not in found:
                        found.add(full)
                        new_here += 1

            print(f"[seeds] {url} -> +{new_here} nuevos (total candidatos esta sesión: {len(found)})")

            if new_here == 0 and page > 1:
                break

            time.sleep(random.uniform(*SLEEP_RANGE))

        if len(found) >= target_needed + 100:
            break

    return list(found)

def extract_article(url):
    soup = get_soup(url)
    if soup is None:
        return None

    h1 = soup.find("h1")
    titulo = h1.get_text(strip=True) if h1 else (
        soup.find("title").get_text(strip=True) if soup.find("title") else "")

    parrafos = soup.find_all("p")
    texto = " ".join(" ".join(p.stripped_strings) for p in parrafos)

    # Limpieza
    texto = re.sub(r"Iniciar sesión para dejar un comentario.*", "", texto, flags=re.IGNORECASE)
    texto = re.sub(r"©\s*El Mundo Today.*", "", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\s+", " ", texto).strip()

    if len(texto) < MIN_TEXT_CHARS:
        return None
    return titulo, texto

def main():
    out_dir = setup_output()
    out_csv = os.path.join(out_dir, "elmundotoday_satire_eval.csv")
    print(f"[INFO] Archivo objetivo: {out_csv}\n")

    # --- FASE 0: CHECKPOINT ---
    existing_urls = set()
    seen_titles = set()
    scraped_count = 0
    file_exists = os.path.exists(out_csv)

    if file_exists:
        try:
            with open(out_csv, "r", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    existing_urls.add(row["url"])
                    seen_titles.add(row["titulo"].lower().strip())
                    scraped_count += 1
            print(f"[CHECKPOINT] Se encontraron {scraped_count} artículos ya guardados en el CSV.")
        except Exception as e:
            print(f"[ERROR] No se pudo leer el CSV existente ({e}). Empezando de cero.")
            file_exists = False
            existing_urls = set()
            seen_titles = set()
            scraped_count = 0
    else:
        print("[CHECKPOINT] No hay CSV previo. Empezando de cero.")

    articles_needed = TARGET_ARTICLES - scraped_count

    if articles_needed <= 0:
        print("\n[OK] ¡Ya has alcanzado o superado el objetivo de 1000 artículos!")
        return

    print(f"[INFO] Faltan {articles_needed} artículos por recolectar.\n")

    # --- FASE 1: ENLACES ---
    print("[1/2] Recolectando enlaces nuevos...")
    links = collect_article_links(existing_urls, articles_needed)
    random.shuffle(links)
    print(f"[INFO] {len(links)} enlaces nuevos candidatos.\n")

    if not links:
        print("[INFO] No se encontraron nuevos enlaces. Terminando.")
        return

    # --- FASE 2: EXTRACCIÓN Y GUARDADO INCREMENTAL ---
    print("[2/2] Extrayendo artículos y guardando en tiempo real...")

    # Abrimos en modo "a" (append) para añadir al final del archivo
    with open(out_csv, "a", newline="", encoding="utf-8") as f:
        fieldnames = ["titulo", "texto", "label", "fuente", "url"]
        w = csv.DictWriter(f, fieldnames=fieldnames)

        # Escribimos cabecera solo si el archivo es nuevo
        if not file_exists:
            w.writeheader()
            file_exists = True # Por si acaso

        session_scraped = 0
        for i, url in enumerate(links, 1):
            if (scraped_count + session_scraped) >= TARGET_ARTICLES:
                break

            res = extract_article(url)
            if res is None:
                continue

            titulo, texto = res
            key = titulo.lower().strip()
            if key in seen_titles:
                continue

            seen_titles.add(key)
            existing_urls.add(url)

            # Escribir la fila directamente en el disco
            w.writerow({
                "titulo": titulo,
                "texto": texto,
                "label": 2,
                "fuente": "ElMundoToday",
                "url": url,
            })

            # ¡CLAVE PARA COLAB! Forzamos la escritura física en Drive
            f.flush()
            os.fsync(f.fileno())

            session_scraped += 1
            total_current = scraped_count + session_scraped

            if session_scraped % 10 == 0 or total_current == TARGET_ARTICLES:
                print(f"  Progreso total: {total_current}/{TARGET_ARTICLES} (Guardados en esta sesión: {session_scraped})")

            time.sleep(random.uniform(*SLEEP_RANGE))

    print(f"\n[OK] Extracción finalizada. Tienes {scraped_count + session_scraped} artículos seguros en Drive.")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[INFO] Archivo objetivo: /content/drive/MyDrive/CorpusNoticiasDeforma/elmundotoday_eval/elmundotoday_satire_eval.csv

[CHECKPOINT] Se encontraron 388 artículos ya guardados en el CSV.
[INFO] Faltan 612 artículos por recolectar.

[1/2] Recolectando enlaces nuevos...
[seeds] https://www.elmundotoday.com/ -> +6 nuevos (total candidatos esta sesión: 6)
[seeds] https://www.elmundotoday.com/page/2/ -> +3 nuevos (total candidatos esta sesión: 9)
[seeds] https://www.elmundotoday.com/page/3/ -> +2 nuevos (total candidatos esta sesión: 11)
[seeds] https://www.elmundotoday.com/page/4/ -> +1 nuevos (total candidatos esta sesión: 12)
[seeds] https://www.elmundotoday.com/page/5/ -> +3 nuevos (total candidatos esta sesión: 15)
[seeds] https://www.elmundotoday.com/page/6/ -> +3 nuevos (total candidatos esta sesión: 18)
[seeds] https://www.elmundotoday.com/page/7/ -> +3 nuevo

Exception ignored from cffi callback <function buffer_callback at 0x7ce503212ca0>:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/curl_cffi/curl.py", line 125, in buffer_callback
    @ffi.def_extern()
    
KeyboardInterrupt: 


  [skip] https://www.elmundotoday.com/2018/11/horoscopo-de-la-semana-leo-lo-sentimos-pasaras-toda-la-semana-intentando-cerrar-un-banner/ (Failed to perform, curl: (23) Failure writing output to destination, passed 13 returned 0. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.)
  Progreso total: 438/1000 (Guardados en esta sesión: 50)
  Progreso total: 448/1000 (Guardados en esta sesión: 60)
  Progreso total: 458/1000 (Guardados en esta sesión: 70)
  Progreso total: 468/1000 (Guardados en esta sesión: 80)
  Progreso total: 478/1000 (Guardados en esta sesión: 90)
  Progreso total: 488/1000 (Guardados en esta sesión: 100)
  Progreso total: 498/1000 (Guardados en esta sesión: 110)
  Progreso total: 508/1000 (Guardados en esta sesión: 120)
  Progreso total: 518/1000 (Guardados en esta sesión: 130)
  Progreso total: 528/1000 (Guardados en esta sesión: 140)
  Progreso total: 538/1000 (Guardados en esta sesión: 150)
  Progreso total: 548/1000 (Guardados en esta sesión